# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets
record_sets = dataset.record_sets()
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print("  Field @ids and names:")
    for f in rs.fields:
        print(f"    - {f.id} (name: {f.name}, type: {f.data_type})")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
all_record_set_ids = [rs.id for rs in dataset.record_sets()]

dataframes = dict()

for record_set_id in all_record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for Record Set @id: '{record_set_id}'")

# Let's choose the main protein data record set for demonstration
# This depends on the actual dataset structure, we'll auto-detect biggest non-empty DataFrame
primary_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
print(f"\nUsing primary record set @id: {primary_record_set_id}")
df = dataframes[primary_record_set_id]
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

In [ ]:
# Identify numeric fields in the chosen record set
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields available: {numeric_fields}\n")

# Select a representative numeric field ('coverage' or 'peptide_count', for example)
numeric_field = None
for field_candidate in ['coverage', 'Coverage', 'peptide_count', 'PeptideCount', 'Abundance', 'MW', 'molecular_weight', 'MW_kDa']:
    if field_candidate in df.columns:
        numeric_field = field_candidate
        break
if numeric_field is None and numeric_fields:
    numeric_field = numeric_fields[0]
print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if numeric_field else 10

filtered_df = df.copy()
if numeric_field:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("\nFirst records with normalized field:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a relevant categorical field
    group_field = None
    for field_candidate in ['description', 'Description', 'Protein', 'gene_symbol', 'Accession', 'accession', 'Modifications']:
        if field_candidate in df.columns:
            group_field = field_candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"\nGrouped mean {numeric_field} by {group_field} (top 5):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If a grouping field is available, plot mean by group (top 10 groups)
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        grouped_df.head(10).plot(kind='bar', color='skyblue')
        plt.title(f'Top 10 {group_field} by mean {numeric_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.tight_layout()
        plt.show()


## 6. Conclusion

- This notebook demonstrated how to load, explore, and analyze a rich proteomics dataset defined by the Croissant FAIR<sup>2</sup> schema using `mlcroissant`.
- Data access by `@id` allows referencing record sets and fields in a reproducible, FAIR manner.
- Basic EDA and visualization can provide initial understanding for downstream proteomics or machine learning workflows.

Explore further: see [Croissant documentation](https://mlcommons.github.io/croissant/) and [dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for schema details.